# TAREFA T1
Lista de variáveis: 

Coeficiente de segurança contra falha por flexão: ${\gamma}_{s} = 2.0$ \
Coeficiente de segurança contra falha por cisalhamento: ${\gamma}_{\tau} = 3.0$ \
Índide de confiabilidade alvo contra falha por flexão: ${\beta}_{sT} = 3.0$ \
Índide de confiabilidade alvo contra falha por cisalhamento: ${\beta}_{\tau T} = 4.0$ \
Custo de falha por flexão: ${c}_{s} = 3.0$ \
Custo de falha por cisalhamento: ${c}_{\tau} = 7.0$ \
Tensão normal devido ao momento fletor: $s = \frac{6M}{bh^2}$\
Tensão cisalhante média: ${\tau} = \frac{3V}{2bh}$ \
Tensão de escoamento em flexão (MPa): $N(\mu_S; \sigma_S) = N(20;2)kNm$ \
Tensão de escoamento em cisalhamento (MPa): $N(\mu_\tau; \sigma_\tau) = N(4;0.4)kNm$ \
Momento solicitante (kNm): $N(\mu_M; \sigma_M) = N(40;8)kNm$ \
Cortante solicitante (kN): $N(\mu_V; \sigma_V) = N(150;30)kNm$ \
Restrição geométrica: $h \le 2b$ 



1) Abordagem DDO \
Temos que a abordagem DDO do problema de minimização da seção da viga submetida a momento fletor e esforço cortante, pode ser escrita como:

$$
\text{determine: } \mathbf{d^*} = \{b^*, h^*\},\\
\text{que minimiza: } f(\mathbf{d}) = bh , \\
\text {sujeito a: } g_s(\mathbf{d}) = 6\mu_M\gamma_S - \mu_Sbh^2 \le 0 , \\
                    g_{\tau(\mathbf{d})} = 3\mu_V\gamma_\tau - 2\mu_{\tau}bh \le 0, \\
                    h - 2b \le 0.

$$


In [44]:
import sympy as sp

# Declarando as variáveis
b, h, mu_M, gamma_S, mu_S, mu_V, f, gamma_tau, mu_tau, = sp.symbols('b, h, mu_M, gamma_S, mu_S, mu_V, f, gamma_tau, mu_tau')
mu_M = 40
gamma_S = 2.0
mu_S = 20.0
mu_V = 150.0
gamma_tau = 3.0
mu_tau = 4.0

# Funções 
f = b*h #Cost function
g_s = (6*mu_M*gamma_S) - (mu_S*b*h**2) # Restrição limite de flexão
g_tau = (3*mu_V*gamma_tau) - (2*mu_tau*b*h) # Restrição limite de esforço cortante
g_a = h-(2*b) # Restrição limite geometrica

Agora precisamos verificar as condições de convexidade das equações apresentadas:

In [62]:
# Verificação da função de restrição de limite de flexão g_s

variables_order_gs = list(ordered(g_s.free_symbols))
hessian_gs = sp.hessian(g_s, variables_order_gs)
auto_valores_hgs = hessian_gs.eigenvals()
print(hessian_gs)
print(auto_valores_hgs)
print("Como a Hessiana não é positiva semi-definida, a restrição g_s não é convexa.")

Matrix([[0, -40.0*h], [-40.0*h, -40.0*b]])
{-20.0*b - 40.0*sqrt(0.25*b**2 + 1.0*h**2): 1, -20.0*b + 40.0*sqrt(0.25*b**2 + 1.0*h**2): 1}
Como a Hessiana não é positiva semi-definida, a restrição g_s não é convexa.


In [63]:
# Verificação da função de restrição de limite de cisalhamento g_tau

variables_order_gtau = list(ordered(g_tau.free_symbols))
hessian_gtau = sp.hessian(g_tau, variables_order_gtau)
auto_valores_hgtau = hessian_gtau.eigenvals()
print(hessian_gtau)
print(auto_valores_hgtau)
print("Como a Hessiana não é positiva semi-definida, a restrição g_tau não é convexa.")

Matrix([[0, -8.00000000000000], [-8.00000000000000, 0]])
{-8.00000000000000: 1, 8.00000000000000: 1}
Como a Hessiana não é positiva semi-definida, a restrição g_tau não é convexa.


In [60]:
# Verificação da função custo

variables_order_f = list(ordered(f.free_symbols))
hessian_f = sp.hessian(f, variables_order_f)
auto_valores_f = hessian_f.eigenvals()
print(hessian_f)
print(auto_valores_f)
print("Como a Hessiana não é positiva semi-definida, a restrição f não é convexa.")

Matrix([[0, 1], [1, 0]])
{-1: 1, 1: 1}
Como a Hessiana não é positiva semi-definida, a restrição f não é convexa.


O resultado do teste de convexidade mostra que não há como garantir a existencia de um minimo global para o problema.\
Agora seguimos com a verificação das condições necessárias de KKT.\
Para isso, escrevemos a função lagrangiana do problema: 
$$
L = f + u_1 (g_s + s_1^2) + u_2 (g_\tau + s_2^2) + u_3 (h - 2b + s_3^2)
$$


In [ ]:
# Vamos escrever a função lagrangiana do problema
u_1, u_2, u_3, s_1, s_2, s_3 = sp.symbols('u_1, u_2, u_3, s_1, s_2, s_3')

L = f + u_1 * (g_s + s_1**2) + u_2 * (g_tau + s_2**2) + u_3 * (g_a + s_3**2) # Função Lagrangiana

# Agora vamos computar todas as derivadas parciais da Lagrangiana
dL_db = sp.diff(L, b)
dL_dh = sp.diff(L, h)
dL_du1 = sp.diff(L, u_1)
dL_du2 = sp.diff(L, u_2)
dL_du3 = sp.diff(L, u_3)
dL_ds1 = sp.diff(L, s_1)
dL_ds2 = sp.diff(L, s_2)
dL_ds3 = sp.diff(L, s_3)

sp.nonlinsolve([dL_db, dL_dh, dL_du1, dL_du2, dL_du3, dL_ds1, dL_ds2, dL_ds3], (u_1, u_2, u_3, s_1, s_2, s_3,b,h))


In [ ]:
# Caso 1 (u_1 = u_2 = u_3 = 0)

valores_u = {u_1: 0, u_2: 0, u_3: 0}
dL_db_new = dL_db.subs(valores_u)
dL_dh_new = dL_dh.subs(valores_u)
sp.solve([dL_db_new, dL_dh_new], (h,b))

{b: 0, h: 0}

In [112]:
# Caso 2 (u_1 = u_2 = 0, s_3 = 0)

valores = {u_1: 0, u_2: 0, s_3: 0}
dL_db_new = dL_db.subs(valores)
dL_dh_new = dL_dh.subs(valores)
sp.solve([dL_db_new, dL_dh_new], (h,b))

{b: -u_3, h: 2*u_3}

In [ ]:
# Caso 3 (u_1 = u_2 = 0, s_3 = 0)

valores = {u_1: 0, u_2: 0, s_3: 0}
dL_db_new = dL_db.subs(valores)
dL_dh_new = dL_dh.subs(valores)
sp.solve([dL_db_new, dL_dh_new], (h,b))